# FSDH Databricks R Sample
*Note: This notebook is a work in progress*

This notebook will use R, but Databricks supports programming in SQL, Scala, and Python as well.

## Connecting to storage
### Option 1: Using Blob storage
To read a file in Databricks, you can use the ABFS (Azure Blob File System). For more information on Azure Blob Storage, see: https://learn.microsoft.com/en-us/azure/storage/blobs/storage-blobs-introduction. We use the sparklyr library to facilitate this in R.


In [0]:
library(sparklyr)
config <- spark_config()
config$sparklyr.databricks.connect <- TRUE

sc <- spark_connect(
  method = "databricks",
  config = config
)
abfss <- "abfss://datahub@fsdhprojdw1poc.dfs.core.windows.net"
df <- spark_read_csv(
  sc,
  name = "df",
  path = paste0(abfss, "/fsdh-sample.csv"),
  header = TRUE
)
head(df, 5)

# Source:   SQL [?? x 5]
# Database: spark_connection
  Name  Sex              Age Height_in Weight_lbs
  <chr> <chr>          <dbl>     <dbl>      <dbl>
1 Alex  "       \"M\""    41        74        170
2 Bert  "       \"M\""    42        68        166
3 Carl  "       \"M\""    32        70        155
4 Dave  "       \"M\""    39        72        167
5 Elly  "       \"F\""    30        66        124

### Option 2: Mount FSDH storage using a storage key
Mounting is only available in Databricks using Python or Scala. If you wish you use mounted storage with R, run the following python code to mount the storage:

In [0]:
%python
if any(mount.mountPoint == "/mnt/fsdh" for mount in dbutils.fs.mounts()):
        dbutils.fs.unmount("/mnt/fsdh")

dbutils.fs.mount(
  source = spark.conf.get('wasbs_uri'),
  mount_point = "/mnt/fsdh",
  extra_configs = {'fs.azure.account.key.' + spark.conf.get('az_storage_name') +'.blob.core.windows.net':dbutils.secrets.get(scope = "datahub", key = "storage-key")})

True

You can now return to R to access the mounted data.

In [0]:
library(sparklyr)

# Connect to Spark
sc <- spark_connect(method = "databricks")

# Read CSV file from mount
df <- spark_read_csv(
  sc,
  name = "fsdh_sample",
  path = "/mnt/fsdh/fsdh-sample.csv",
  header = TRUE
)

# Show first 5 rows
df %>% head(5)

# Source:   SQL [?? x 5]
# Database: spark_connection
  Name  Sex              Age Height_in Weight_lbs
  <chr> <chr>          <dbl>     <dbl>      <dbl>
1 Alex  "       \"M\""    41        74        170
2 Bert  "       \"M\""    42        68        166
3 Carl  "       \"M\""    32        70        155
4 Dave  "       \"M\""    39        72        167
5 Elly  "       \"F\""    30        66        124

## Further resources
For more help with Databricks, consult the [Resources section](https://poc.fsdh-dhsf.science.cloud-nuage.canada.ca/resources/) of the Federal Science DataHub.